In [1]:
import os, gc, joblib
import numpy as np, pandas as pd
import pyarrow.dataset as pads
import lightgbm as lgb
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

DATA = "/home/ilya/pyspark-project/data_parquet"
TRAIN_END = "2026-04-01"      # published before this -> training; after -> holdout check
                              # set to "2100-01-01" to train on everything (no holdout)
OUT_DIR = "model_artifacts"
KAPPA, SEED = 10, 0
SUPPORTED = ["epNotificationEF2020", "epNotificationEZK2020", "epNotificationEZT2020"]

P_COLS = ["purchase_number", "purchase_type", "responsible", "published_at_ts",
          "collecting_finished_at_ts", "max_price", "contract_guarantee_part",
          "ktru_cnt", "okpd2_cnt", "position_numbers_cnt", "object_info"]
RAW_COLS = P_COLS + ["okpd2", "ktru"]          # schema of a raw notice

first = lambda a: a[0] if a is not None and len(a) > 0 else None

def add_codes(df):
    """first OKPD2 code, KTRU fallback, 2- and 5-char product keys"""
    df["okpd2_1"] = df["okpd2"].map(first)
    df["ktru_1"] = df["ktru"].map(first)
    code = df["okpd2_1"].fillna(df["ktru_1"])
    df["prod2"], df["prod5"] = code.str[:2], code.str[:5]
    return df

# ---- notices ----
p = pads.dataset(f"{DATA}/purchases_flat", format="parquet").to_table(columns=RAW_COLS).to_pandas()
p = add_codes(p).drop(columns=["okpd2", "ktru"])

# ---- outcomes ----
w = pads.dataset(f"{DATA}/winner_price_flat", format="parquet").to_table(
    columns=["purchase_number", "winner_price", "winner_protocol_published_at", "application_block"]).to_pandas()
w = w[w["application_block"] != "epProtocolEF2020SubmitOffers"]

d = p.merge(w, on="purchase_number")
del p, w; gc.collect()
d = d[d["purchase_type"].isin(SUPPORTED) & (d["max_price"] > 1)].reset_index(drop=True)
d["ratio"] = d["winner_price"] / d["max_price"]
d["proto"] = pd.to_datetime(d["winner_protocol_published_at"], format="ISO8601").astype("datetime64[ns]")

is_tr = (d["published_at_ts"] < TRAIN_END).values
CATS = {"purchase_type": pd.Index(sorted(SUPPORTED)),
        "okpd2_2": pd.Index(sorted(d["okpd2_1"].str[:2].dropna().unique()))}
CAP = d.loc[is_tr, "max_price"].quantile(0.999)      # cap for the ruble-weighted metric

def base_features(df):
    X = pd.DataFrame(index=df.index)
    X["purchase_type_c"] = pd.Categorical(df["purchase_type"], categories=CATS["purchase_type"])
    X["log_max"] = np.log1p(df["max_price"])
    X["contract_guarantee_part"] = df["contract_guarantee_part"]
    lead = (df["collecting_finished_at_ts"] - df["published_at_ts"]).dt.total_seconds() / 86400
    X["lead_days"] = lead.where(lead >= 0).clip(upper=40)
    for c in ["ktru_cnt", "okpd2_cnt", "position_numbers_cnt"]:
        X[c] = df[c]
    X["okpd2_2_c"] = pd.Categorical(df["okpd2_1"].str[:2], categories=CATS["okpd2_2"])
    X["txt_len"] = df["object_info"].str.len()
    return X

print("labeled notices:", len(d), "| train:", int(is_tr.sum()), "| holdout:", int((~is_tr).sum()))

labeled notices: 2873806 | train: 2524985 | holdout: 348821


In [2]:
obj = d["object_info"].fillna("")
tr_idx = np.where(is_tr)[0]
rs = np.random.RandomState(SEED)

vec = TfidfVectorizer(min_df=20, max_features=50000, ngram_range=(1, 2), sublinear_tf=True)
vec.fit(obj.iloc[rs.choice(tr_idx, min(400_000, len(tr_idx)), replace=False)])
X_txt = vec.transform(obj)

targets = {"low": (d["ratio"] < 0.01).values, "nodrop": (d["ratio"] >= 0.99999).values}
blk = pd.qcut(d.loc[is_tr, "published_at_ts"].rank(method="first").values, 4, labels=False)

def fit_lr(ix, y):
    ix = rs.choice(ix, min(300_000, len(ix)), replace=False)
    return LogisticRegression(C=2, max_iter=200).fit(X_txt[ix], y[ix])

text_models, txt = {}, {}
for nm, y in targets.items():
    s = np.full(len(d), np.nan)
    for b in range(4):                                   # out-of-fold, in time blocks
        inb, outb = tr_idx[blk == b], tr_idx[blk != b]
        s[inb] = fit_lr(outb, y).predict_proba(X_txt[inb])[:, 1]
    text_models[nm] = fit_lr(tr_idx, y)                  # final model: used for holdout and live notices
    rest = np.where(~is_tr)[0]
    if len(rest):
        s[rest] = text_models[nm].predict_proba(X_txt[rest])[:, 1]
    txt[f"txt_{nm}"] = s
    print(nm, "done")

del X_txt; gc.collect()

low done
nodrop done


20

In [3]:
hcols = ["r_n", "r_low", "r_nodrop", "r_ratio"]
s2c = ["s2_n", "s2_low", "s2_nodrop", "s2_ratio"]
s5c = ["s5_n", "s5_low", "s5_nodrop", "s5_ratio"]

store = pd.DataFrame({
    "purchase_number": d["purchase_number"].values,
    "responsible": d["responsible"].values, "ptype": d["purchase_type"].values,
    "prod2": d["prod2"].values, "prod5": d["prod5"].values, "proto": d["proto"].values,
    "low": (d["ratio"] < 0.01).astype(float).values,
    "nodrop": (d["ratio"] >= 0.99999).astype(float).values,
    "ratio": d["ratio"].values})
GM = store.loc[is_tr, ["low", "nodrop", "ratio"]].mean().to_numpy()   # global prior

def make_keys(resp, ptype, prod2, prod5):
    resp = pd.Series(np.asarray(resp, dtype=object))
    isna = resp.isna().to_numpy()
    r = resp.astype(str).to_numpy(dtype=object)
    pt = pd.Series(np.asarray(ptype, dtype=object)).astype(str).to_numpy(dtype=object)
    base = r + "|" + pt + "|"
    k2 = base + pd.Series(np.asarray(prod2, dtype=object)).fillna("NA").to_numpy(dtype=object)
    k5 = base + pd.Series(np.asarray(prod5, dtype=object)).fillna("NA").to_numpy(dtype=object)
    return np.where(isna, "__NA__", r).astype(object), k2, k5

def build_index(keys):
    vals = np.c_[np.ones(len(store)), store[["low", "nodrop", "ratio"]].to_numpy()]
    prt = store["proto"].to_numpy(dtype="datetime64[ns]")
    out = {}
    for k, ix in pd.Series(np.arange(len(store))).groupby(np.asarray(keys, dtype=object)).indices.items():
        o = ix[np.argsort(prt[ix])]
        out[k] = (prt[o], np.vstack([np.zeros((1, 4)), np.cumsum(vals[o], axis=0)]))
    return out

def build_all_indexes():
    ko, k2, k5 = make_keys(store["responsible"], store["ptype"], store["prod2"], store["prod5"])
    idx = {"org": build_index(ko), "s2": build_index(k2), "s5": build_index(k5)}
    idx["org"].pop("__NA__", None)
    return idx

def lookup(index, keys, pub):
    n = np.zeros(len(keys)); s = np.zeros((len(keys), 3))
    for k, ix in pd.Series(np.arange(len(keys))).groupby(np.asarray(keys, dtype=object)).indices.items():
        if k in index:
            pr, cs = index[k]
            k1 = np.searchsorted(pr, pub[ix], side="left")    # only protocols strictly before publication
            n[ix] = cs[k1, 0]; s[ix] = cs[k1, 1:]
    return n, s

def history_features(resp, ptype, prod2, prod5, pub):
    ko, k2, k5 = make_keys(resp, ptype, prod2, prod5)
    n0, s0 = lookup(IDX["org"], ko, pub)
    r0 = np.where(n0[:, None] > 0, s0 / np.maximum(n0, 1)[:, None], np.nan)
    org = np.where(np.isnan(r0), GM, r0)
    n2, s2 = lookup(IDX["s2"], k2, pub); r2 = (s2 + KAPPA * org) / (n2 + KAPPA)[:, None]
    n5, s5 = lookup(IDX["s5"], k5, pub); r5 = (s5 + KAPPA * r2) / (n5 + KAPPA)[:, None]
    return pd.DataFrame(np.c_[n0, r0, n2, r2, n5, r5], columns=hcols + s2c + s5c)

IDX = build_all_indexes()
H = history_features(d["responsible"], d["purchase_type"], d["prod2"], d["prod5"],
                     d["published_at_ts"].to_numpy(dtype="datetime64[ns]"))
print("history built:", H.shape)

history built: (2873806, 12)


In [4]:
FEATS = ["purchase_type_c", "log_max", "contract_guarantee_part", "lead_days", "ktru_cnt", "okpd2_cnt",
         "position_numbers_cnt", "okpd2_2_c", "txt_len", "txt_low", "txt_nodrop"] + hcols + s2c + s5c

Xall = pd.concat([base_features(d), pd.DataFrame(txt), H], axis=1)[FEATS]
Xtr, ytr = Xall[is_tr], d.loc[is_tr, "ratio"].values
sw = np.sqrt(d.loc[is_tr, "max_price"].values)
cls = np.select([ytr < 0.01, ytr >= 0.99999], [0, 2], 1)      # 0 = per-unit/low, 1 = partial drop, 2 = no drop
mid = cls == 1

def mk(obj, n):
    return lgb.LGBMRegressor(objective=obj, n_estimators=n, learning_rate=0.03, num_leaves=31,
                             min_child_samples=1000, subsample=0.8, subsample_freq=1, colsample_bytree=0.7,
                             reg_lambda=5, cat_smooth=50, min_data_per_group=500,
                             random_state=SEED, verbose=-1)

clf = lgb.LGBMClassifier(objective="multiclass", n_estimators=1500, learning_rate=0.03, num_leaves=31,
                         min_child_samples=1000, subsample=0.8, subsample_freq=1, colsample_bytree=0.7,
                         reg_lambda=5, random_state=SEED, verbose=-1).fit(Xtr, cls, sample_weight=sw)
print("classifier done")
reg = mk("l1", 1500).fit(Xtr[mid], ytr[mid], sample_weight=sw[mid])      # ratio given a partial drop
print("mid regression done")
l2 = mk("l2", 2000).fit(Xtr, ytr, sample_weight=sw)                     # mean head, for totals
print("L2 head done")

classifier done
mid regression done
L2 head done


In [5]:
def mix_median(P, r):
    grid = np.linspace(0, 1, 201)
    cdf = P[:, [0]] * (grid >= 0.002) + P[:, [1]] * (grid[None, :] >= r[:, None])
    med = grid[(cdf >= 0.5).argmax(1)]
    return np.where(P[:, 0] + P[:, 1] < 0.5, 1.0, med)

def predict_X(X):
    Pk = clf.predict_proba(X)
    med = mix_median(Pk, np.clip(reg.predict(X), 0, 1))
    mean = np.clip(l2.predict(X), 0, 1)
    return Pk, med, mean

def build_X(raw):
    raw = add_codes(raw.reset_index(drop=True).copy())
    X = base_features(raw)
    Xt = vec.transform(raw["object_info"].fillna(""))
    X["txt_low"] = text_models["low"].predict_proba(Xt)[:, 1]
    X["txt_nodrop"] = text_models["nodrop"].predict_proba(Xt)[:, 1]
    H_ = history_features(raw["responsible"], raw["purchase_type"], raw["prod2"], raw["prod5"],
                          raw["published_at_ts"].to_numpy(dtype="datetime64[ns]"))
    return pd.concat([X, H_], axis=1)[FEATS]

def predict(raw):
    """raw: notices in the purchases_flat schema (RAW_COLS). Returns predicted winner prices."""
    raw = raw[raw["purchase_type"].isin(SUPPORTED) & (raw["max_price"] > 1)].reset_index(drop=True)
    Pk, med, mean = predict_X(build_X(raw))
    out = raw[["purchase_number", "max_price"]].copy()
    out["p_low"], out["p_nodrop"] = Pk[:, 0], Pk[:, 2]
    out["ratio_median"], out["ratio_mean"] = med, mean
    out["predicted_price"] = med * out["max_price"]        # one contract
    out["predicted_price_for_sums"] = mean * out["max_price"]  # budgets / totals
    return out

# ---------------- holdout evaluation ----------------
ho = ~is_tr
if ho.any():
    hd = d[ho].reset_index(drop=True)
    Pk, med, mean = predict_X(Xall[ho])
    y, mp = hd["ratio"].values, hd["max_price"].values
    cw = np.minimum(mp, CAP)
    base_med = hd["purchase_type"].map(d.loc[is_tr].groupby("purchase_type")["ratio"].median()).values
    print(f"{'':22s} {'MAE':>7s} {'ruble-w capped':>15s} {'total-spend err':>16s}")
    for nm, pr in [("baseline (median/type)", base_med), ("median head", med), ("mean head", mean)]:
        e = np.abs(y - pr)
        print(f"{nm:22s} {e.mean():7.4f} {np.average(e, weights=cw):15.4f} "
              f"{(pr * mp).sum() / (y * mp).sum() - 1:16.3%}")

    hd["predicted_price"], hd["actual_price"] = med * mp, y * mp
    hd["error_%"] = (hd["predicted_price"] - hd["actual_price"]) / hd["actual_price"] * 100
    ape = (hd["predicted_price"] - hd["actual_price"]).abs() / hd["actual_price"]
    print(f"\nwithin 10% of true price: {(ape <= .10).mean():.3f} | within 25%: {(ape <= .25).mean():.3f}")
    pd.set_option("display.width", 200, "display.float_format", "{:,.0f}".format)
    hd["object"] = hd["object_info"].str.slice(0, 40)
    print(hd.sample(15, random_state=42)[["object", "max_price", "predicted_price", "actual_price", "error_%"]]
          .to_string(index=False))

    # parity: the raw-notice path must reproduce the training-path predictions
    ids = hd.sample(5000, random_state=1)["purchase_number"].tolist()
    raw = pads.dataset(f"{DATA}/purchases_flat", format="parquet").to_table(
        columns=RAW_COLS, filter=pads.field("purchase_number").isin(ids)).to_pandas()
    chk = predict(raw).merge(pd.DataFrame({"purchase_number": hd["purchase_number"], "ref": med}),
                             on="purchase_number")
    print("\nparity, max abs diff in ratio_median:", float((chk["ratio_median"] - chk["ref"]).abs().max()))

                           MAE  ruble-w capped  total-spend err
baseline (median/type)  0.2073          0.2078          15.837%
median head             0.1494          0.1363           1.137%
mean head               0.1574          0.1506          -0.294%

within 10% of true price: 0.462 | within 25%: 0.678
                                  object  max_price  predicted_price  actual_price  error_%
Поставка топлива  автомобильного с автоз  1,060,000            5,300           234    2,161
    Поставка ковра борцовского (12х12 м)    921,180          732,338       714,180        3
Выполнение работ по ремонту кузова автом    158,667          105,513       158,667      -34
                                     NaN     51,520           50,232        51,450       -2
Закупка изделий кухонных  из нержавеющей     51,592           33,277        51,592      -36
Поставка систем инфузионных (Реестровый     215,400            1,077            75    1,345
   Система хранения данных (в сфере ИКТ)  4,618

In [6]:
os.makedirs(OUT_DIR, exist_ok=True)
joblib.dump({"clf": clf, "reg": reg, "l2": l2, "features": FEATS, "vec": vec, "text_models": text_models,
             "cats": CATS, "gm": GM, "cap": CAP, "store": store}, f"{OUT_DIR}/predictor.joblib")

def update_store(raw, winners):
    """Append newly labeled purchases to the history and rebuild the indexes.
    raw: notices (RAW_COLS); winners: purchase_number, winner_price, winner_protocol_published_at, application_block"""
    global store, IDX
    x = add_codes(raw.copy()).merge(winners, on="purchase_number")
    x = x[(x["application_block"] != "epProtocolEF2020SubmitOffers") & (x["max_price"] > 1)
          & x["purchase_type"].isin(SUPPORTED) & ~x["purchase_number"].isin(store["purchase_number"])]
    ratio = x["winner_price"] / x["max_price"]
    new = pd.DataFrame({
        "purchase_number": x["purchase_number"].values, "responsible": x["responsible"].values,
        "ptype": x["purchase_type"].values, "prod2": x["prod2"].values, "prod5": x["prod5"].values,
        "proto": pd.to_datetime(x["winner_protocol_published_at"], format="ISO8601").astype("datetime64[ns]").values,
        "low": (ratio < 0.01).astype(float).values, "nodrop": (ratio >= 0.99999).astype(float).values,
        "ratio": ratio.values})
    store = pd.concat([store, new], ignore_index=True)
    IDX = build_all_indexes()
    print("store size:", len(store))